In [0]:
process_name='pricing_data_migration_bronze'
Folder_name='daily_pricing'
Landing_Layer_Folder_Path="/Volumes/pricing_analytics/storageac/pricing_analytics/landing/daily-pricing/"
bronze_Layer_Folder_Path=f"/Volumes/pricing_analytics/storageac/pricing_analytics/bronze/{Folder_name}/"
quarntine_Layer_Folder_Path=f"/Volumes/pricing_analytics/storageac/pricing_analytics/quarntine/{Folder_name}/"
bronze_catalog_path=f"pricing_analytics.bronze.{Folder_name}"
pipeline_control_log="pricing_analytics.bronze.pipeline_control_logs"

In [0]:
# find out unprocessed file dates

processed_date=spark.sql(f"""select date(process_table_date_time) from {pipeline_control_log} where process_name='{process_name}' and process_status='Completed'""").collect()

bronze_layer_processed_file_dates=[str(names.process_table_date_time) for names in processed_date]

landing_layer_processed_file_dates=[Files.name.replace('/', '') for Files in dbutils.fs.ls(Landing_Layer_Folder_Path)]

un_processed_file_dates=list((set(landing_layer_processed_file_dates)-set(bronze_layer_processed_file_dates)))

if len(un_processed_file_dates)==0:
    dbutils.notebook.exit('There is no new files to process')


In [0]:
import re

from pyspark.sql.functions import col, current_timestamp

from pyspark.sql.types import StringType

expected_file_format=r'^PW_MW_DR_\d{8}\.csv$'

expected_column=['DATE_OF_PRICING', 'ROW_ID', 'MODAL_PRICE']

for file_date in un_processed_file_dates:
    valid_files=[]
    for unprocessed_files in dbutils.fs.ls(Landing_Layer_Folder_Path+'/'+file_date):
        file_status='Pass'
        fail_reason=None
        if not unprocessed_files.name.lower().endswith('.csv'):
            file_status='Fail'
            fail_reason='The File not endswith csv'

        elif unprocessed_files.size==0:
            file_status='Fail' 
            fail_reason='The file size is zero'

        elif not re.match(expected_file_format, unprocessed_files.name):
            file_status='Fail'
            fail_reason='The file not matching with expected name'

        else:
            try:
                test = spark.read.format("csv").option("header", True).load(unprocessed_files.path).limit(1)
                if not set(expected_column).issubset(test.columns):
                    file_status='Fail'
                    fail_reason='The File not contain primary keys'
            except Exception:
                file_status='Fail'
                fail_reason='The File not readable'
        
        if file_status=='Fail':
            dbutils.fs.mv(unprocessed_files.path, quarntine_Layer_Folder_Path+file_date+'/'+unprocessed_files.name)
        
        else:
            valid_files.append(unprocessed_files.path)
        
        spark.sql(f"""insert into pricing_analytics.bronze.file_process_control_logs values(
            '{process_name}',
            '{file_date}',
            '{unprocessed_files.name}',
            'Completed',
            '{fail_reason}',
            current_timestamp()
            )""") 
    if len(valid_files)>0:
        df=spark.read.format('csv').option('header', True).load(valid_files)
        string_df = df.select([col(c).cast(StringType()).alias(c) for c in df.columns])
        final_df=string_df.withColumn("File_inserted_date", current_timestamp()).withColumn("File_updated_date", current_timestamp())

        final_df.write.format('delta').mode("append").option("mergeSchema", True).save(bronze_Layer_Folder_Path)

        final_df.write.format("delta").mode('append').option('mergeSchema', True).saveAsTable(bronze_catalog_path)

        spark.sql(f"""insert into pricing_analytics.bronze.pipeline_control_logs values(
            '{process_name}',
            '{file_date}',
            'Completed',
            current_timestamp()
        )""")  